# Proportional - Integrative - Derivative (PID) Controllers: Generalities

## Introduction

A PID (proportional, integral, derivative) controller is an algorithm extensively used in industrial systems to generate a sequence of actuator commands ($u(t)$) over time ($t$) to make the system output ($y(t)$) match a desired setpoint value ($r(t)$), and it does so using only the tracking error ($e(t)$): the instantaneous difference between the setpoint and a measurement of the system output ($y_m(t)$). PID control is so widely applicable and successful, that it sometimes referred to as "the most successful technology of all times".  

The secret to PID success lies in several outstanding properties: 

- The control signal ($u(t)$) is intuitively generated by the simple sum of three terms: a proportional (to the tracking error), an integral (to the tracking error), and a derivative (to the tracking error) one;
- It does not require an in-depth knowledge of the system (model) that is being controlled, as long as a measurement of the output is available;
- It is simple to implement, both digitally and analogically;
- It is easy to tune (but difficult to do so "well");

The weaknesses of PID control are significant too:

- It is in no sense optimal;
- The is no general "golden rule" to tune it perfectly, but some structured guidelines area available (one is reported in this notebook: the Ziegler-Nichols method);
- There are no guarantees of robustness, especially for more complex dynamical systems;

## Fundamental notions of PID control

Let us us start from defining useful terminology and listing facts of PID control. 

### Terminology and Definitions

When referring to PID control, the following terms are typically used: 

* **System**: typically a dynamic process that, provided input signals ($u(t)$), produces output signals ($y(t)$). We can consider it a black box at this stage. 
* **Process Variable/Measured Output**: The parameter of the system that is being measured and controlled. It is typically indicated with $y_m(t)$, to distinguish it from the "true" system output $y(t)$. For more information on basic control systems notations: [Duckietown introduction to Control Systems](https://docs.duckietown.com/ente/duckietown-manual/80-instructor-manual/available-resources/slides/control/01-controls-intro.html).
* **Setpoint/Reference signal**: is the desired value $r(t)$ we are trying to drive the process variable to.
* **Control Variable/Manipulated Variable**: The output of the controller that serves as input to the system, to drive the error between the setpoint and the process variable down over time.
* **Steady-State Value**: The value of a signal as time goes to infinity. It might exist (e.g., when a system is stable and converges over time), or not (variable continue oscillating over time between different values). 
* **Steady-State Error**: The difference between the setpoint and the steady-state output signal value.
* **Rise Time**: The time required for the process variable to rise from 10 percent to 90 percent of the steady-state value.
* **Settling Time**: The time required for the process variable to settle within a certain percentage of the steady-state value.
* **Overshoot**: The maximum amount the process variable exceeds the setpoint (expressed as a percentage).

### General PID Algorithm

The tracking error $e(t)$, is ideally calculated as the difference between the setpoint $r(t)$ and the process variable, or system output, $y(t) = f(x,u,t)$; where $f(\cdot)$ is a shortcut way to represent "whatever system (dynamic process) is going on between the input and output of the system". 

Therefore, always ideally, the tracking error is:

$$e(t) = r(t) - y(t)$$

We are insisting on the "ideal" part because the _true_ value of the process variable is generally unknown, all we can do is measure it. Measuring implies the use of some instrument, which always introduces measurement noise of some kind. Truly, therefore, any practical PID implementation is driven by $e(t) = r(t) - y_m(t)$.

When tuned appropriately, the PID controller improves the _performance_ and _robustness_ of the closed loop response, by reducing the rise time and settling time of the system, eliminating steady-state error, introducing some degree of disturbance rejection, and improving stability. It does so by changing the control variable $u(t)$ based on three control terms.

#### Proportional Term: the present 

The first control term is the proportional term, which produces an output that is directly proportional to $e(t)$:

$$P(t) = K_pe(t)$$

The magnitude of the proportional response is dependent upon $K_p \in \mathcal{R}$, which is the "proportional gain" constant. A higher proportional gain constant indicates a greater change in the controller's output in response to the system's error at the current time.

#### Integral Term: the past

The integral term accounts for the accumulated tracking error over time. The output produced by this term is the sum of the instantaneous error over a certain time window, multiplied by the "integral gain constant" $K_i \in \mathcal{R}$:

$$I(t) = K_i\int_{t_0}^t\!e(\tau)\,\mathrm{d}\tau$$

#### Derivative Term: the future

The derivative term is determined by the rate of change of the system's error over time, similarly multiplied by the "derivative gain constant" $K_d \in \mathcal{R}$:

$$D(t) = K_d\frac{de(t)}{dt}$$

#### Together, the PID controller

The PID control function is the sum of the proportional, integral, and derivative terms:

$$u_{PID}(t) = P(t) + I(t) + D(t) = K_pe(t) + K_i\int_{k_0}^t\!e(\tau)\,\mathrm{d}\tau + K_d\frac{de(t)}{dt}$$

The figure below summarizes the inclusion of a PID controller within a basic control loop.

<figure>
    <img src="../assets/_images/pid/pid_controller_block_diagram.png" alt = "PID Control Block Diagram"  style="width:80%">
</figure>

In most practical applications, especially the digital ones such as robotics, the *time discretized* form of the control function is actually implemented, where $t = t_k = kT_{step}, \quad k=1,2,... \quad T_{step >0}$:

$$u(t_k) = K_pe(t_k) + K_i\sum_{i=k_0}^k e(t_i)\Delta t + K_d\frac{e(t_k)-e(t_{k-1})}{\Delta t} $$



Entire books have been written on the effect of time discretization in control systems, including of course on PID controllers. Let us just say, for the purpose of this application, that _discretization_ is an approximation of the beautiful perfection of mathematical _continuity_, an assumption underlying all the classical literature. Therefore, it leads to imperfections, or errors (with respect to the easier to analyze time-continuous case). The larger $T_{step}$, i.e., the bigger the discretization, the larger the approximation error.

### Tuning the PID controller parameters

"Tuning" of a PID controller refers to the process of iterating through the values of the $K_p$, $K_i$, and $K_d$ parameters to obtain a desireable (closed loop, i.e., the system with the controller active and "plugged in") control response. 

After understanding the general effects of each control term on the closed-loop response, tuning can be accomplished through educated trial-and-error or by other specialized tuning schemes, such as the [Ziegler-Nichols tuning method](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method). 

Although the independent effects of each parameter are explained below, the three control terms may be correlated and so changing one parameter may impact the influence of another. The general effects of each term are therefore useful as reference, but the actual effects will vary depending on the specific dynamics of the underlying system, in other words, what is the exact form of what we previously referred to as $f(\cdot)$. 

Control literature, and us too in this notebook, tends to focus on explaning the effects of the PID parameters on the closed-loop system response assuming the controlled system is a second-order LTI one, i.e., (i) linear, and (ii) time-invariant, (iii) defined by a second order differential equation. Delving in the definition of LTI systems is beyond the scope of this LX (the curious reader can refer to a typical [Control System I introductory class on modeling and linearization](https://www.youtube.com/watch?v=GTiHWvx9N-4&list=PLP-dxWt_NmHtqCxuTAmzNVTPAgXossvUc&index=2&t=2174s)), but a few things are worth keeping in mind to make sense of these apparently arbitrary assumptions:

1. Real-world system are pretty much all **not** LTI systems, but rather nonlinear systems;
2. All nonlinear systems can be ("locally") approximated as LTI systems; 
3. The predominant long-term time response of dynamic system of any order can be approximated through the "dominant poles" assumption to a second-order system. 

In summary, none of what follows is _fundamentally_ true, but it is close enough that it actually works in practice (most of the times, with sufficient patience in tuning)! 

#### Effects of $K_p$

For a given $e(t)$, increasing $K_p$ will proportionally increase the control output. This generally causes the system to react more quickly and aggressively, decreasing the rise time ("getting there quicker") significantly and settling time by a small amount. On the other hand, increasing the proportional gain tends to cause overshoot, which in turn could destabilize the system. 

Increasing $K_p$ also has the effect of decreasing the steady-state error. However, as the value of the process variable approaches the setpoint and the error decreases, the proportional term will also decrease. As a result, with a P-controller (a controller with only the proportional term, i.e., ($K_i = K_d = 0$), the process variable will asymptotically approach the setpoint, but will never quite reach it. Thus, a P-controller cannot be used to completely eliminate steady-state error.

In summary, the P part of the controller reacts based only on the present, situation, and it is good practice to start PID tuning from adjusting $K_p$ until the system shows "decent" performance, typically indicated by getting to the setpoint quickly, overshooting a little, and continuing oscillating around it.

#### Effects of $K_i$

The integral term takes into account the history of the error, i.e., its persistance over time. 

The longer the error is different from zero, the larger the integral term will grow, eventually driving the error down. Integral control has the most notable effect of reducing and eliminating steady-state error and introducing disturbance rejection to a certain class of external signals. However, the build-up of error can cause the value of the process variable to overshoot, which can increase the settling time of the system, though it decreases the rise time. Moreover, implementing an integral term on a real system, through a computer or microcontroller, requires memory storage.

Integral control, in practical implementations, has a few additional caveats to keep in mind. The most notable one is related to the so called "windup" challenge, which is discussed below along with "anti-windup" approaches.

#### Effects of $K_d$

By calculating the instantaneous rate of change of the system's error, the derivative term provides an approximate measure of the future tracking error, anticipating what is to come, and making the sytem more reactive rather than proactive. 

While the proportional and integral terms both act to move the process variable to the setpoint, the derivative term rather has the effect of dampening their efforts and decreasing the amount the system overshoots. If appropriately tuned, the derivative term reduces oscillations, overshoot, settling time and improves the stability of the system. The derivative term has negligible effects on steady-state error and only decreases the rise time by a minor amount.

A **very important caveat** of derivative terms is the following: *never take derivatives of noisy signals*. If there is one thing you want to remember from this whole notebook is this. 

The most common form of noise, white noise, which we have learned to be practically present in every real-world PID controller, will yield a close to infinite value when derived, leading to unreasonable $u_{PID}(t)$, system instability, and most likely to things breaking and/or people getting hurt. 

Therefore, the best pratices in using derivative terms in real-world PID are:

1. always double-check the quality of the error signal before turning on the derivative term, in particular how noisy it is; 
2. [Low-pass filter](https://en.wikipedia.org/wiki/Low-pass_filter) the signal, if needed, before sending it to the controller, or otherwise remove/mitigate the high frequency oscillations. Applying averaging algorithms (e.g., moving windows) are a popular alternative;
3. When tuning, always start with very tiny (with respect to the other two terms in the controller) $K_d$ values, and verify that $D(t)$ is in the order of magnitude of $P(t)$ and $I(t)$ _before_ activating the controller on the real system/plant.

Finally, derivative (as well as integral) terms require, as their name suggests, derivatives (and integrals). Those who are familiar with calculus, the branch of math that explores integrals and derivatives, might recall that this mathematical operation (yes, derivatives and integrals are fundamentally the same operation, like addition and subtraction) underlies the notion of _infinity_ (infinitely large, or infinitely small:_infinitesimal_). 

Infinity is a very qualitative notion. Computers tend to work better with quantitative inputs rather than qualitative guidelines, hence practically calculating these terms in a real-world scenario leads to design choices and inevitable approximation errors. Think about it:

* What is infinite (-ly large)? The formal definition is: pick the biggest number you can think of, _infinite_ is strictly more than that
* What is infinitesimal (infinitely small)? Pick a positive number, the smallest you can think of (the famous $/epsilon$), _infinitesimal_ is stricly smaller than that

### Calculating integrals in discrete time

$I(t)$ is defined as proportional to the integral of the error over time:

$$ e_{int}(t) = k_p \int_{t_0}^t e(\tau) d\tau$$

An integral is an _infinite_ **sum** of _infinitesimal_  bits, a concept that assumes _continuity_ of time. Continuity means that, for any two instant in time you can think about, arbitrarily close to each other, there will always be _infinite_ other instant between them.

But all computers in the world, including the one running on the Duckiebot, don't know how to do infinite or infinitesimal. "Think of the smallest" and "think of the biggest" are qualitative concepts that only humans can grasp.

Computers can do finite (instead of infinite or infinitesimal) though. Time, for a robot, is a sequence of instants. But when you make a computer take two consecutive instants, there is nothing in between. Computers have a notion of _discrete_ time, not continuous time.

The immediate repercussion of this fundamental limitation of computers is that we cannot really calculate integrals.

What we can do is have them calculate a _finite sum_ to approximate the actual integral. 

$$ e_{int}(t) = k_p \int_0^t e(\tau) d\tau \simeq \sum_{i=k_0}^{k} e_i \Delta t = (e_0 + e_1 + \dots + e_{k-1} + e_k)\Delta t$$

Where in the above approximation we assumed for simplicity that all time instants are equally spaced by a constant _time step_ $T_{step}=\Delta t=t_k-t_{k-1}$.

So how do we implment this integral component on our Duckietown robots? We can note that:

$$ e_{int,k}= (e_0 + e_1 + \dots + e_{k-1} + e_k)\Delta t = (e_0 + e_1 + \dots + e_{k-1})\Delta t + e_k\Delta t = e_{int,k-1} + e_k \Delta t.$$

and implement this form in a function, as will be shown later in this LX. 

### Calculating derivatives in discrete time

Derivatives are the tool math uses to measure change. As you might imagine, derivatives are tremendously important operations as so many things in the universe change in some way. One could argue that derivatives are the most important of operators, and in fact open the doors to a whole fields of math.  

Time derivatives are defined as _the ratio of the difference of a function, evaluated at two infinitesimally close to each other instants, and the time difference between them_. 

In fancy words, derivatives are _limits of the incremental ratio_ functions:

$$ \dot e_t = \frac{de_t}{dt} = \lim_{dt \rightarrow 0} \frac{e_{t+dt}-e_t}{dt}$$

Without getting in the details, that $\lim$ part means that the $dt$ time difference is a very, very, small (positive) number. How small? The smaller you can imagine it the better you understood the derivative operation. _At the limit, it's zero_.  

But computers cannot do this "at the limit", because it is a qualitative leap of the mind. This is a very human thing to do. Computer only know finite time. So they need to know how much is actually $dt$ equal to (0.001? 0.0000001? or maybe even smaller that that?).

Computer cannot do derivatives, they can do _finite differences_, which approximate derivatives. There are many different formulations of finite differences, the simplest is called "Euler backwards" method. It basically approximates the derivative as a difference of the current and the previous function evaluation, divided by the time step $\Delta T = t_{k}-t_{k-1}$. 

$$\frac{de_t}{dt} \simeq e_{der,k} = \frac{e_k - e_{k-1}}{\Delta T}.$$


### Summary

<figure>
    <img src="../assets/_images/pid/control_term_effects_table.png" alt = "Summary of PID controller terms effect"  style="width:80%">
</figure>


### Ziegler-Nichols Closed-Loop Tuning Method
Ziegler and Nichols developed [two techniques](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method) for tuning PID controllers: a closed-loop tuning method and an open-loop tuning method. With the closed-loop tuning method, the PID controller is initially turned into a P controller with $K_p$ set to zero. $K_p$ is slowly increased until the system exhibits stable oscillatory behavior, at which point it is denoted $K_u$, the ultimate or critical gain. As such, $K_u$ should be the smallest $K_p$ value that causes the control loop to have regular oscillations. The ultimate or critical period $T_u$ of the oscillations needs to be measured. Then, using the constants determined experimentally by Ziegler and Nichols, the controller gain values can be computed as follows:  

$$ K_p = 0.6K_u $$
$$ K_i = 2K_p / (T_u) $$
$$ K_d = K_p(T_u) / 8 $$

Although the Ziegler-Nichols method may yield initial tuning values that work relatively well, the system's control loop can be tuned further by adjusting the controller gain values based on the general effects of each control term as explained above.

### Theory meeting the real world: the "nuisances"
In real-world applications, the PID controller exhibits issues that require modifications to the general algorithm. In certain situations, one may find that a P-Controller, PD-Controller (eliminating the integral term), or a PI-Controller (eliminating the derivative term) are more advantageous controllers for the system. Additionally, various techniques can be employed to counteract common implementation challenges.

#### Integral Windup
Integral wind-up occurs when, due to a large change in setpoint, the control output causes the system's actuator to become saturated. At this point, the integrated error between the process variable and the setpoint will continue to grow (because the actuator is at its limit and cannot drive the process variable any closer to the setpoint). In turn, the control output will continue to grow and will no longer have any effect on the system. When the setpoint finally changes and the error changes sign (meaning the new setpoint is now below the value of the process variable), the integral term will take a while to "unwind" all of the error that it has accumulated before producing a reverse control action that will move the process variable in the correct direction towards the setpoint.

There exist numerous ways to address integral wind-up. One way is to keep the integral term within predefined upper and lower bounds. Another way is to set the integral term to zero if the control output will cause the system's actuator to saturate. Yet another way is to reduce the integral term by a constant multiplied by the difference between the actual output and the commanded output. If the actuator is not saturated, then the difference between the actual and commanded output will be zero and will not affect the integral term. If the actuator is saturated, then the additional feedback in the control loop will drive the commanded output closer to the saturation limit. If the setpoint changes and causes the error to change sign, then the integral term will not need to unwind in order to produce an appropriate control action. Setpoint ramping &mdash; in which the setpoint is increased or decreased incrementally to reach the desired value &mdash; may also help prevent integral wind-up.

#### Derivative Noise

As previously discussed but worth repeating, derivative terms are potentially dangerous and should be handled with care. Since the derivative term is proportional to the change in error, it is highly sensitive to noise (which can produce drastic changes in error during consecutive time steps / measurements). Using a low-pass filter on the derivative term,taking a weighted mean of previous derivative terms, or applying averages of the measured process can help ensure that high-frequency noise does not cause the derivative term to skyrocket numerically and adversely affect the control output. Safety hard upper bounds to the derivative terms can be applied, just in case.  

### Cascaded Controllers

When multiple measurements can be used to control a single process variable, these measurements can be combined using a cascaded PID controller. In cascaded PID control, two PID controllers are used conjointly to yield a better control response. The output of the PID controller for the outer control loop determines the setpoint for the PID controller of the inner control loop. The outer loop controller controls the primary process variable of the system, while the inner loop controller controls a system parameter that tends to change more rapidly in order to minimize the error of the outer control loop. The two controllers have separate tuning values, which can be optimized for the part of the system that they control. This enables an overall better control response for the system as a whole. We will see that the Duckiedrone implements a number of cojoined PID loops. 


# Activity


## The True Value and Error Curves
The figure below shows a true value curve for a PID controller. Draw the corresponding error curve for this graph. You can draw by hand and upload the picture. (Hint: refer to the error definition equation from before)

<figure>
  <img src="../assets/_images/pid/true_value_curve.png" alt="True value curve" style="width:60%">
  <figcaption>True Value Curve for A PID Controller. The orange dot line indicates the setpoint and the black line is the true value curve.</figcaption>
</figure>


## Explain an Effect

Answer the following questions (3-5 sentences each):
  * What will happen when the absolute value of $K_{p}$ is very large? What will happen when the absolute value of $K_{p}$ is very small?
  * Can $K_{p}$ be tuned such that the $P$ term stops oscillations? Why or why not?
  * Can the process variable stabilize at the setpoint (i.e. zero steady-state error) with only the $P$ term and the $D$ term? Why or why not?                                      

<!-- 
[ANSWERS]
    The rise time decreases when $K_{p}$ increases.
    The settling time increases when $K_{i}$ increases.
    The overshoot decreases when $K_{d}$ increases.                                   
-->   

Explain the following effects caused by $K_{p}$, $K_{i}$ and $K_{d}$ (3-5 sentences each). For example, here is a sample answer (though you do not need to follow the pattern):

  * [Q:] *The rise time decreases when $K_{d}$ increases.
  * [A:] *When $K_{d}$ increases, the error at time step $t+1$ decreases. This is because larger and larger $K_{d}$ results in larger and larger control signals at time step $t$. This drives the system to achieve a lower error at time step $t+1$. As the error at time step $t+1$ decreases, the slope of the true value curve increases. Since the slope increases, the rising time towards the setpoint should decrease (slightly).




## Start Tuning

When designing a PID controller, it is important to choose a good set of $K_{p}$, $K_{i}$, and $K_{d}$; poor choices can result in undesirable behavior. The graphs in the figure below illustrate behavior resulting from unknown sets of $K_{p}$, $K_{i}$, and $K_{d}$. In each graph, the orange dot line indicates the setpoint and the black line is the true value curve. For each graph, answer the following (1-2 sentences each):


1. Which term(s) went wrong, if any? In other words, which term(s) are too high or too low?
2. How can you correct the behavior?

<figure>
    <img src="../assets/_images/pid/tuning1.png" alt = "PID tuning option"  style="width:60%">
</figure>

<figure>
    <img src="../assets/_images/pid/tuning2.png" alt = "PID tuning option"  style="width:60%">
</figure>

<figure>
    <img src="../assets/_images/pid/tuning3.png" alt = "PID tuning option"  style="width:60%">
</figure>

<figure>
    <img src="../assets/_images/pid/tuning4.png" alt = "PID tuning option"  style="width:60%">
</figure>



